# Hombro y codo · movimientos independientes
**Mateo Flores y Federico Soler — Grupo 3 — Introducción a la Robótica**

Esta versión muestra **hombro → húmero → codo → antebrazo → muñeca**, con **una articulación en movimiento por vez**. La muñeca es el efector final y no tiene otro servo. La vista lateral permite entender la flexión del brazo en un plano vertical, sin plataforma giratoria.

**Para reproducir:** elegir **Entorno de ejecución → Ejecutar todas** y luego **▶**. La captura real de Wokwi ya está incluida: **363 registros y 7,343 segundos**, con cero cambios simultáneos de ambas consignas. No se requiere subir archivos para la primera reproducción.

El nuevo sketch y el nuevo diagrama se ejecutaron en una sesión temporal de Wokwi. El código corresponde a esta versión del hombro y el codo; los datos no provienen del proyecto anterior de base giratoria. El proyecto está publicado en https://github.com/federicosolerup/brazo-robotico-arduino-wokwi. No se guardaron cambios en el proyecto anterior de Wokwi.

## 1. Secuencia y convención de ángulos

| Tramo | Hombro | Codo | Articulación que se mueve |
|---|---:|---:|---|
| Reposo | 30° | 30° | Ninguna |
| 1 | 30° → 75° | Fijo en 30° | Hombro |
| 2 | Fijo en 75° | 30° → 90° | Codo |
| 3 | 75° → 30° | Fijo en 90° | Hombro |
| 4 | Fijo en 30° | 90° → 30° | Codo |

El hombro se mide respecto de la horizontal. El codo se mide **respecto del húmero**, con 0° como extensión completa. Al girar el hombro, el codo se desplaza en el espacio y el antebrazo acompaña al húmero, pero la flexión del codo permanece constante. Al girar el codo, el húmero permanece quieto.

Cada tramo utiliza el perfil $q(t)=q_0+(q_f-q_0)(3s^2-2s^3)$, con $s=t/T$ y $T=1500\,\mathrm{ms}$, y termina con una pausa de 300 ms. El servo que no se mueve mantiene su consigna; no se desconecta durante el ciclo.

## 2. Cinemática directa

La altura ilustrativa del hombro es $h=12\,\mathrm{cm}$, el húmero mide $L_1=18\,\mathrm{cm}$ y el antebrazo $L_2=15\,\mathrm{cm}$.

$$x_c=L_1\cos q_1,\qquad z_c=h+L_1\sin q_1$$
$$x_m=L_1\cos q_1+L_2\cos(q_1+q_2),\qquad z_m=h+L_1\sin q_1+L_2\sin(q_1+q_2)$$

El ángulo absoluto del antebrazo es $q_1+q_2$; su ángulo relativo al húmero es $q_2$. Las longitudes son ilustrativas. El modelo representa consignas angulares y no calcula fuerzas, torque ni errores de seguimiento físico.

In [ ]:
# @title Código de lectura, verificación y visualización (expandir para inspeccionar)
"""Brazo de hombro y codo con movimientos articulares alternados.

python visualizar_brazo.py --csv datos/captura_wokwi.csv --gif --mp4
Sin --csv se utiliza una demostración teórica identificada como tal.
"""
from pathlib import Path
import argparse
import csv
import io
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from matplotlib.patches import Circle, Arc, Polygon, FancyBboxPatch

PUNTOS = np.array([[30, 30], [75, 30], [75, 90], [30, 90], [30, 30]])
FASES = ['hombro_1', 'codo_2', 'hombro_3', 'codo_4']
ALTURA_CM = 12.0
HUMERO_CM = 18.0
ANTEBRAZO_CM = 15.0
AZUL = '#1C6DA1'
NARANJA = '#DA7841'
VERDE = '#17846B'
TINTA = '#21394A'
GRIS = '#738795'
FONDO = '#F6F8FA'


def cargar_csv(texto, ciclo=None, origen='CSV importado · consignas de Arduino'):
    """Lee encabezados seriales, reinicios, comentarios y varios ciclos."""
    ansi = re.compile(r'\x1b\[[0-?]*[ -/]*[@-~]')
    requeridas = {'tiempo_ms', 'hombro_deg', 'codo_deg'}
    encabezado, sesion, grupos = None, -1, {}
    for numero, linea in enumerate(texto.splitlines(), 1):
        linea = ansi.sub('', linea).strip().lstrip('\ufeff')
        if not linea or linea.startswith('#'):
            continue
        campos = [v.strip() for v in next(csv.reader([linea]))]
        if requeridas.issubset(campos):
            encabezado, sesion = campos, sesion+1
            continue
        if encabezado is None:
            continue
        if len(campos) != len(encabezado):
            raise ValueError(f'Línea {numero}: fila incompleta.')
        fila = dict(zip(encabezado, campos))
        try:
            c = int(fila.get('ciclo', '1'))
            t, h, e = [float(fila[k]) for k in ['tiempo_ms', 'hombro_deg', 'codo_deg']]
        except ValueError as exc:
            raise ValueError(f'Línea {numero}: valores no numéricos.') from exc
        if not np.isfinite([t, h, e]).all() or t < 0 or not (0 <= h <= 180 and 0 <= e <= 180):
            raise ValueError(f'Línea {numero}: valores fuera de rango.')
        grupos.setdefault((sesion, c), []).append((t, h, e, fila.get('fase', 'registro')))
    elegibles = [k for k in grupos if ciclo is None or k[1] == ciclo]
    if not elegibles:
        raise ValueError('Falta una captura válida: usar tiempo_ms,hombro_deg,codo_deg y un ciclo completo.')
    clave = elegibles[-1]
    filas = grupos[clave]
    if any(b[0] < a[0] for a, b in zip(filas, filas[1:])):
        raise ValueError('El tiempo retrocede dentro de un ciclo. Revisar la captura.')
    # No ocultar un cambio simultáneo ocurrido incluso en dos filas del mismo ms.
    originales = np.array([f[:3] for f in filas], dtype=float)
    cambios = np.abs(np.diff(originales[:, 1:3], axis=0)) > 1e-9
    if np.any(cambios[:, 0] & cambios[:, 1]):
        raise ValueError('La captura contiene cambios simultáneos de hombro y codo.')
    filas = list({f[0]: f for f in filas}.values())
    if len(filas) < 2:
        raise ValueError('Se necesitan al menos dos instantes distintos.')
    v = np.array([f[:3] for f in filas], dtype=float)
    return {'t': (v[:, 0]-v[0, 0])/1000, 'hombro': v[:, 1], 'codo': v[:, 2],
            'fase': np.array([f[3] for f in filas]), 'origen': origen,
            'ciclo': clave[1], 'sesion': clave[0]+1}


def datos_demostracion():
    """Calcula datos de ejemplo; no son una ejecución del simulador."""
    filas, reloj = [], 0
    for i, (a, b) in enumerate(zip(PUNTOS[:-1], PUNTOS[1:])):
        for ms in range(0, 1501, 20):
            s = ms/1500
            q = (a+(b-a)*(3*s*s-2*s*s*s)).astype(int)
            filas.append([1, reloj+ms, *q, FASES[i]])
        reloj += 1520
        for ms in range(0, 301, 20):
            filas.append([1, reloj+ms, *b, f'pausa_{i+1}'])
        reloj += 300
    filas.append([1, reloj, 30, 30, 'fin'])
    texto = io.StringIO()
    w = csv.writer(texto, lineterminator='\n')
    w.writerow(['ciclo', 'tiempo_ms', 'hombro_deg', 'codo_deg', 'fase'])
    w.writerows(filas)
    return cargar_csv(texto.getvalue(), origen='DEMOSTRACIÓN TEÓRICA · sin captura de Wokwi')


def cinematica(hombro, codo, altura=ALTURA_CM, humero=HUMERO_CM, antebrazo=ANTEBRAZO_CM):
    """Cinemática 2R en el plano vertical XZ, con hombro fijo en (0, altura).

    Hombro: ángulo absoluto del húmero respecto de +X.
    Codo: flexión relativa entre húmero prolongado y antebrazo; 0° = extendido.
    """
    q1, q2 = np.broadcast_arrays(np.deg2rad(hombro), np.deg2rad(codo))
    e = np.stack([humero*np.cos(q1), altura+humero*np.sin(q1)], axis=-1)
    w = e+np.stack([antebrazo*np.cos(q1+q2), antebrazo*np.sin(q1+q2)], axis=-1)
    return e, w


def verificar_independencia(datos):
    q = np.column_stack([datos['hombro'], datos['codo']])
    cambios = np.abs(np.diff(q, axis=0)) > 1e-9
    simultaneos = int(np.sum(cambios[:, 0] & cambios[:, 1]))
    tramos = {}
    for i, fase in enumerate(FASES):
        sel = np.flatnonzero(datos['fase'] == fase)
        activo = 0 if fase.startswith('hombro') else 1
        correcto = len(sel) > 1 and bool(np.allclose(q[sel, 1-activo], PUNTOS[i, 1-activo]))
        correcto = correcto and bool(np.allclose(q[sel[0]], PUNTOS[i])) and bool(np.allclose(q[sel[-1]], PUNTOS[i+1])) if len(sel) else False
        tramos[fase] = {'articulacion_activa': 'hombro' if activo == 0 else 'codo',
                        'otra_articulacion_fija_y_extremos_correctos': bool(correcto)}
    cursor, encontrados = 0, []
    for objetivo in PUNTOS:
        indices = np.flatnonzero(np.all(np.isclose(q[cursor:], objetivo), axis=1))
        if not len(indices):
            break
        cursor += int(indices[0]); encontrados.append(float(datos['t'][cursor])); cursor += 1
    return {'origen': datos['origen'], 'muestras': len(q),
            'duracion_s': round(float(datos['t'][-1]), 3),
            'cambios_simultaneos': simultaneos,
            'articulaciones_independientes': simultaneos == 0,
            'configuraciones_en_orden': len(encontrados) == 5,
            'regresa_a_reposo': bool(np.allclose(q[-1], PUNTOS[0])), 'tramos': tramos}


def crear_animacion(datos, fps=20):
    if not 1 <= fps <= 60:
        raise ValueError('FPS debe estar entre 1 y 60.')
    r = verificar_independencia(datos)
    if r['cambios_simultaneos']:
        raise ValueError('Este proyecto requiere una articulación por vez.')
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                         'text.color': TINTA, 'axes.labelcolor': TINTA,
                         'xtick.color': GRIS, 'ytick.color': GRIS})
    fig = plt.figure(figsize=(14, 8), facecolor=FONDO)
    fig.text(.045, .94, 'Hombro y codo', fontsize=28, weight='bold')
    fig.text(.045, .9, 'GRUPO 3   /   Mateo Flores · Federico Soler', fontsize=10, color=GRIS)
    fig.text(.955, .943, 'UNA ARTICULACIÓN POR VEZ', ha='right', fontsize=12, weight='bold', color=VERDE)
    fig.text(.955, .903, datos['origen'], ha='right', fontsize=9, color=GRIS)
    ax = fig.add_axes([.045, .19, .56, .64], facecolor=FONDO)
    ax.set(xlim=(-22, 35), ylim=(0, 48), xlabel='Alcance horizontal · X (cm)', ylabel='Altura · Z (cm)')
    ax.set_aspect('equal'); ax.set_xticks([-20, 0, 10, 20, 30]); ax.set_yticks([0, 12, 24, 36, 48])
    ax.grid(color='#E2E8ED', lw=.7, zorder=0)
    ax.spines[['top', 'right']].set_visible(False)
    # Torso y cabeza fijos: no hay una plataforma ni giro de base.
    ax.add_patch(Polygon([(-7, 0), (-8, 13), (-5, 17), (-2, 16), (0, 12), (-1.5, 0)],
                         facecolor='#DCE3E9', edgecolor='#B6C3CE', lw=1.3, zorder=2))
    ax.add_patch(Circle((-5.1, 21), 3, facecolor='#E5EBEF', edgecolor='#B6C3CE', lw=1.3))
    ax.text(-7, 4, 'Torso\nfijo', fontsize=9, color=GRIS, ha='center')
    ax.plot([-15, 30], [0, 0], color='#B6C3CE', lw=2)
    e, w = cinematica(datos['hombro'], datos['codo'])
    ax.plot(w[:, 0], w[:, 1], color='#D0DDE5', lw=1.2, zorder=1)
    (rastro,) = ax.plot([], [], color=VERDE, lw=1.6, alpha=.65, zorder=2)
    (humero_borde,) = ax.plot([], [], color='white', lw=24, solid_capstyle='round', zorder=3)
    (humero,) = ax.plot([], [], color=AZUL, lw=18, solid_capstyle='round', zorder=4)
    (ante_borde,) = ax.plot([], [], color='white', lw=22, solid_capstyle='round', zorder=3)
    (antebrazo,) = ax.plot([], [], color=NARANJA, lw=16, solid_capstyle='round', zorder=4)
    aro_h = Circle((0, ALTURA_CM), 1.7, facecolor='white', edgecolor=AZUL, lw=2.5, zorder=6)
    aro_c = Circle((0, 0), 1.6, facecolor='white', edgecolor=NARANJA, lw=2.5, zorder=6)
    ax.add_patch(aro_h); ax.add_patch(aro_c)
    ax.plot([0], [ALTURA_CM], 'o', color=AZUL, ms=5, zorder=7)
    (centro_c,) = ax.plot([], [], 'o', color=NARANJA, ms=5, zorder=7)
    (muneca,) = ax.plot([], [], 'o', ms=10, color=VERDE, mec='white', mew=2, zorder=7)
    arco_h = Arc((0, ALTURA_CM), 11, 11, theta1=0, theta2=30, color=AZUL, lw=1.5)
    arco_c = Arc((0, 0), 9, 9, theta1=30, theta2=60, color=NARANJA, lw=1.5)
    ax.add_patch(arco_h); ax.add_patch(arco_c)
    (referencia,) = ax.plot([], [], color=NARANJA, ls=':', lw=1, zorder=1)
    h_label = ax.annotate('HÚMERO', xy=(0, 0), xytext=(-18, 31), fontsize=10,
                          color=AZUL, weight='bold', zorder=9,
                          arrowprops={'arrowstyle': '-', 'color': AZUL, 'lw': .9})
    a_label = ax.annotate('ANTEBRAZO', xy=(0, 0), xytext=(20, 45), fontsize=10,
                          color=NARANJA, weight='bold', ha='center', zorder=9,
                          arrowprops={'arrowstyle': '-', 'color': NARANJA, 'lw': .9})
    c_label = ax.text(0, 0, 'Codo · D10', fontsize=9, color=TINTA)
    w_label = ax.text(0, 0, 'Muñeca\nEfector final', fontsize=9, color=VERDE)
    ax.annotate('Hombro · D9', (0, ALTURA_CM), (-17, 9), fontsize=9, color=TINTA,
                arrowprops={'arrowstyle': '-', 'color': GRIS, 'lw': .8})
    graf = fig.add_axes([.66, .415, .30, .285], facecolor='white')
    graf.plot(datos['t'], datos['hombro'], color=AZUL, lw=2.3, drawstyle='steps-post', label='Hombro · D9')
    graf.plot(datos['t'], datos['codo'], color=NARANJA, lw=2.3, drawstyle='steps-post', label='Codo · D10')
    graf.set(xlim=(0, max(float(datos['t'][-1]), .1)), ylim=(20, 105),
             xlabel='Tiempo (s)', ylabel='Ángulo articular (°)')
    graf.set_yticks([30, 60, 75, 90]); graf.grid(alpha=.13)
    graf.spines[['top', 'right']].set_visible(False)
    graf.legend(frameon=False, loc='upper left', fontsize=9)
    cursor = graf.axvline(0, color=TINTA, ls='--', lw=1)
    (p_h,) = graf.plot([], [], 'o', ms=6, color=AZUL)
    (p_c,) = graf.plot([], [], 'o', ms=6, color=NARANJA)
    estado = fig.text(.66, .82, '', fontsize=16, weight='bold')
    detalle = fig.text(.66, .78, '', fontsize=10, color=GRIS)
    reloj = fig.text(.66, .735, '', fontsize=10, color=GRIS)
    angulos = fig.text(.66, .32, '', fontsize=17, weight='bold')
    fig.add_artist(FancyBboxPatch((.655, .215), .305, .06, transform=fig.transFigure,
                                 boxstyle='round,pad=0.012', facecolor='#E6F2ED', edgecolor='none'))
    fig.text(.67, .238, 'CAMBIOS SIMULTÁNEOS: 0', color=VERDE, fontsize=11, weight='bold')
    fig.text(.045, .12, 'HOMBRO → HÚMERO → CODO → ANTEBRAZO → MUÑECA', color=TINTA, fontsize=10, weight='bold')
    fig.text(.045, .079, 'Vista lateral · plano vertical. El codo se mide respecto del húmero; 0° significa brazo extendido.', color=GRIS, fontsize=9)
    fig.text(.045, .047, 'Dimensiones ilustrativas: húmero 18 cm y antebrazo 15 cm. La muñeca es el efector, sin otro actuador.', color=GRIS, fontsize=9)
    instantes = np.r_[np.arange(0, datos['t'][-1], 1/fps), datos['t'][-1]]

    def actualizar(t):
        i = max(0, min(int(np.searchsorted(datos['t'], t, side='right')-1), len(datos['t'])-1))
        elbow, wrist = e[i], w[i]
        q1, q2 = datos['hombro'][i], datos['codo'][i]
        fase = datos['fase'][i]
        h_activo, c_activo = fase.startswith('hombro_'), fase.startswith('codo_')
        for line in [humero_borde, humero]: line.set_data([0, elbow[0]], [ALTURA_CM, elbow[1]])
        for line in [ante_borde, antebrazo]: line.set_data([elbow[0], wrist[0]], [elbow[1], wrist[1]])
        aro_c.center = elbow
        aro_h.set_linewidth(4 if h_activo else 2)
        aro_c.set_linewidth(4 if c_activo else 2)
        centro_c.set_data([elbow[0]], [elbow[1]])
        muneca.set_data([wrist[0]], [wrist[1]])
        rastro.set_data(w[:i+1, 0], w[:i+1, 1])
        arco_h.theta2 = q1
        arco_c.center = elbow; arco_c.theta1 = q1; arco_c.theta2 = q1+q2
        u = np.array([np.cos(np.deg2rad(q1)), np.sin(np.deg2rad(q1))])
        referencia.set_data([elbow[0], elbow[0]+5*u[0]], [elbow[1], elbow[1]+5*u[1]])
        h_label.xy = (elbow[0]/2, (ALTURA_CM+elbow[1])/2)
        a_label.xy = ((elbow[0]+wrist[0])/2, (elbow[1]+wrist[1])/2)
        c_label.set_position((elbow[0]+2, elbow[1]-2))
        w_label.set_position((wrist[0]+2, wrist[1]-.5))
        cursor.set_xdata([t, t]); p_h.set_data([t], [q1]); p_c.set_data([t], [q2])
        if h_activo:
            estado.set_text('Mueve el hombro'); estado.set_color(AZUL)
            detalle.set_text(f'Codo fijo en {q2:.0f}° · acompaña al húmero')
        elif c_activo:
            estado.set_text('Mueve el codo'); estado.set_color(NARANJA)
            detalle.set_text(f'Hombro fijo en {q1:.0f}° · húmero quieto')
        else:
            estado.set_text('Reposo final' if fase == 'fin' else 'Pausa · ambas articulaciones fijas')
            estado.set_color(TINTA); detalle.set_text('Se mantienen las dos consignas angulares')
        reloj.set_text(f't = {t:.2f} s   /   {datos["t"][-1]:.2f} s')
        angulos.set_text(f'Hombro {q1:.0f}°    Codo {q2:.0f}°')
        return humero, antebrazo, muneca, cursor, estado

    actualizar(0)
    anim = FuncAnimation(fig, actualizar, frames=instantes, interval=1000/fps, blit=False, repeat=False)
    anim.actualizar_instante = actualizar
    return fig, anim


def guardar_html(anim, destino, fps=20):
    with plt.rc_context({'animation.frame_format': 'jpeg', 'savefig.dpi': 90, 'animation.embed_limit': 80}):
        reproductor = anim.to_jshtml(fps=fps, default_mode='once')
    reproductor = re.sub(r'<link rel="stylesheet"[^>]*font-awesome[^>]*>', '', reproductor)
    reproductor = reproductor.replace('<i class="fa fa-play fa-flip-horizontal"></i>', '◀')
    iconos = {'fa-fast-backward': '⏮', 'fa-step-backward': '◀', 'fa-backward': '◁',
              'fa-pause': 'Ⅱ', 'fa-play': '▶', 'fa-step-forward': '▶', 'fa-fast-forward': '⏭',
              'fa-minus': '−', 'fa-plus': '+'}
    for k, v in iconos.items(): reproductor = reproductor.replace(f'<i class="fa {k}"></i>', v)
    pagina = ('<!doctype html><html lang="es"><meta charset="utf-8">'
              '<meta name="viewport" content="width=device-width, initial-scale=1">'
              '<title>Hombro y codo · movimientos independientes</title>'
              '<style>body{margin:0;background:#F6F8FA;font-family:Arial,sans-serif}'
              '.animation{margin:auto;max-width:1400px}.animation img{max-width:100%;height:auto}'
              'button,input,select{font-size:15px}</style>'+reproductor+'</html>')
    Path(destino).write_text(pagina, encoding='utf-8')

## 3. Elegir los datos

- `captura_incluida`: ejecución real de esta versión en Wokwi.
- `subir_csv`: cargar un nuevo CSV desde el equipo.
- `pegar_csv`: pegar la salida serial con su encabezado en `CSV_PEGADO`.
- `demostracion`: generar datos teóricos identificados como tales.

El encabezado debe incluir `tiempo_ms,hombro_deg,codo_deg`. El CSV de la versión anterior de base/hombro no corresponde a este modelo. Si hay varios ciclos se toma el último por defecto; `CICLO` permite elegir otro. Se rechazan valores inválidos y cambios simultáneos de ambos ángulos.

In [ ]:
# @title Parámetros
MODO = "captura_incluida"  # @param ["captura_incluida", "subir_csv", "pegar_csv", "demostracion"]
CICLO = None
FPS = 20
CSV_PEGADO = """
"""

In [ ]:
# @title Captura incluida y procedencia
CSV_INCLUIDO = 'ciclo,tiempo_ms,hombro_deg,codo_deg,fase\n1,0,30,30,inicio\n1,1,30,30,hombro_1\n1,21,30,30,hombro_1\n1,42,30,30,hombro_1\n1,62,30,30,hombro_1\n1,84,30,30,hombro_1\n1,104,30,30,hombro_1\n1,125,30,30,hombro_1\n1,145,31,30,hombro_1\n1,166,31,30,hombro_1\n1,187,31,30,hombro_1\n1,208,32,30,hombro_1\n1,228,32,30,hombro_1\n1,249,33,30,hombro_1\n1,270,33,30,hombro_1\n1,291,34,30,hombro_1\n1,311,34,30,hombro_1\n1,332,35,30,hombro_1\n1,353,36,30,hombro_1\n1,374,36,30,hombro_1\n1,394,37,30,hombro_1\n1,415,38,30,hombro_1\n1,436,39,30,hombro_1\n1,457,39,30,hombro_1\n1,477,40,30,hombro_1\n1,498,41,30,hombro_1\n1,518,42,30,hombro_1\n1,540,43,30,hombro_1\n1,560,44,30,hombro_1\n1,581,44,30,hombro_1\n1,601,45,30,hombro_1\n1,622,46,30,hombro_1\n1,643,47,30,hombro_1\n1,663,48,30,hombro_1\n1,684,49,30,hombro_1\n1,705,50,30,hombro_1\n1,726,51,30,hombro_1\n1,746,52,30,hombro_1\n1,767,53,30,hombro_1\n1,788,54,30,hombro_1\n1,809,55,30,hombro_1\n1,829,55,30,hombro_1\n1,850,56,30,hombro_1\n1,870,57,30,hombro_1\n1,892,58,30,hombro_1\n1,912,59,30,hombro_1\n1,933,60,30,hombro_1\n1,953,61,30,hombro_1\n1,975,62,30,hombro_1\n1,995,63,30,hombro_1\n1,1016,63,30,hombro_1\n1,1036,64,30,hombro_1\n1,1058,65,30,hombro_1\n1,1078,66,30,hombro_1\n1,1099,67,30,hombro_1\n1,1119,67,30,hombro_1\n1,1141,68,30,hombro_1\n1,1161,69,30,hombro_1\n1,1182,69,30,hombro_1\n1,1203,70,30,hombro_1\n1,1224,70,30,hombro_1\n1,1244,71,30,hombro_1\n1,1265,72,30,hombro_1\n1,1286,72,30,hombro_1\n1,1307,72,30,hombro_1\n1,1327,73,30,hombro_1\n1,1347,73,30,hombro_1\n1,1369,74,30,hombro_1\n1,1389,74,30,hombro_1\n1,1410,74,30,hombro_1\n1,1430,74,30,hombro_1\n1,1452,74,30,hombro_1\n1,1472,74,30,hombro_1\n1,1493,74,30,hombro_1\n1,1514,75,30,hombro_1\n1,1535,75,30,pausa_1\n1,1555,75,30,pausa_1\n1,1576,75,30,pausa_1\n1,1596,75,30,pausa_1\n1,1617,75,30,pausa_1\n1,1637,75,30,pausa_1\n1,1658,75,30,pausa_1\n1,1678,75,30,pausa_1\n1,1699,75,30,pausa_1\n1,1720,75,30,pausa_1\n1,1741,75,30,pausa_1\n1,1761,75,30,pausa_1\n1,1782,75,30,pausa_1\n1,1802,75,30,pausa_1\n1,1823,75,30,pausa_1\n1,1834,75,30,pausa_1\n1,1835,75,30,pausa_1\n1,1836,75,30,codo_2\n1,1856,75,30,codo_2\n1,1877,75,30,codo_2\n1,1898,75,30,codo_2\n1,1919,75,30,codo_2\n1,1939,75,30,codo_2\n1,1960,75,31,codo_2\n1,1981,75,31,codo_2\n1,2002,75,32,codo_2\n1,2022,75,32,codo_2\n1,2043,75,33,codo_2\n1,2064,75,33,codo_2\n1,2085,75,34,codo_2\n1,2105,75,35,codo_2\n1,2126,75,35,codo_2\n1,2147,75,36,codo_2\n1,2168,75,37,codo_2\n1,2188,75,38,codo_2\n1,2209,75,39,codo_2\n1,2230,75,40,codo_2\n1,2251,75,41,codo_2\n1,2271,75,42,codo_2\n1,2292,75,43,codo_2\n1,2313,75,44,codo_2\n1,2334,75,45,codo_2\n1,2354,75,46,codo_2\n1,2375,75,47,codo_2\n1,2396,75,48,codo_2\n1,2417,75,50,codo_2\n1,2437,75,51,codo_2\n1,2457,75,52,codo_2\n1,2479,75,53,codo_2\n1,2499,75,54,codo_2\n1,2520,75,56,codo_2\n1,2540,75,57,codo_2\n1,2562,75,58,codo_2\n1,2582,75,59,codo_2\n1,2603,75,61,codo_2\n1,2624,75,62,codo_2\n1,2645,75,63,codo_2\n1,2665,75,64,codo_2\n1,2686,75,65,codo_2\n1,2707,75,67,codo_2\n1,2728,75,68,codo_2\n1,2748,75,69,codo_2\n1,2769,75,70,codo_2\n1,2790,75,71,codo_2\n1,2811,75,73,codo_2\n1,2831,75,74,codo_2\n1,2852,75,75,codo_2\n1,2873,75,76,codo_2\n1,2894,75,77,codo_2\n1,2914,75,78,codo_2\n1,2935,75,79,codo_2\n1,2956,75,80,codo_2\n1,2977,75,81,codo_2\n1,2997,75,82,codo_2\n1,3018,75,83,codo_2\n1,3039,75,83,codo_2\n1,3060,75,84,codo_2\n1,3080,75,85,codo_2\n1,3101,75,86,codo_2\n1,3122,75,86,codo_2\n1,3143,75,87,codo_2\n1,3163,75,87,codo_2\n1,3184,75,88,codo_2\n1,3205,75,88,codo_2\n1,3225,75,89,codo_2\n1,3246,75,89,codo_2\n1,3266,75,89,codo_2\n1,3288,75,89,codo_2\n1,3308,75,89,codo_2\n1,3329,75,89,codo_2\n1,3349,75,90,codo_2\n1,3371,75,90,pausa_2\n1,3391,75,90,pausa_2\n1,3412,75,90,pausa_2\n1,3432,75,90,pausa_2\n1,3453,75,90,pausa_2\n1,3473,75,90,pausa_2\n1,3494,75,90,pausa_2\n1,3514,75,90,pausa_2\n1,3535,75,90,pausa_2\n1,3555,75,90,pausa_2\n1,3577,75,90,pausa_2\n1,3597,75,90,pausa_2\n1,3618,75,90,pausa_2\n1,3638,75,90,pausa_2\n1,3659,75,90,pausa_2\n1,3670,75,90,pausa_2\n1,3671,75,90,hombro_3\n1,3691,74,90,hombro_3\n1,3713,74,90,hombro_3\n1,3733,74,90,hombro_3\n1,3754,74,90,hombro_3\n1,3774,74,90,hombro_3\n1,3796,74,90,hombro_3\n1,3816,73,90,hombro_3\n1,3837,73,90,hombro_3\n1,3857,73,90,hombro_3\n1,3879,72,90,hombro_3\n1,3899,72,90,hombro_3\n1,3920,71,90,hombro_3\n1,3941,71,90,hombro_3\n1,3962,70,90,hombro_3\n1,3982,69,90,hombro_3\n1,4003,69,90,hombro_3\n1,4024,68,90,hombro_3\n1,4045,68,90,hombro_3\n1,4065,67,90,hombro_3\n1,4087,66,90,hombro_3\n1,4107,65,90,hombro_3\n1,4128,65,90,hombro_3\n1,4148,64,90,hombro_3\n1,4170,63,90,hombro_3\n1,4190,62,90,hombro_3\n1,4211,61,90,hombro_3\n1,4231,60,90,hombro_3\n1,4253,59,90,hombro_3\n1,4273,59,90,hombro_3\n1,4294,58,90,hombro_3\n1,4314,57,90,hombro_3\n1,4336,56,90,hombro_3\n1,4356,55,90,hombro_3\n1,4376,54,90,hombro_3\n1,4398,53,90,hombro_3\n1,4418,52,90,hombro_3\n1,4439,51,90,hombro_3\n1,4459,50,90,hombro_3\n1,4481,49,90,hombro_3\n1,4501,48,90,hombro_3\n1,4522,47,90,hombro_3\n1,4543,47,90,hombro_3\n1,4564,46,90,hombro_3\n1,4584,45,90,hombro_3\n1,4605,44,90,hombro_3\n1,4626,43,90,hombro_3\n1,4647,42,90,hombro_3\n1,4667,41,90,hombro_3\n1,4688,40,90,hombro_3\n1,4709,40,90,hombro_3\n1,4730,39,90,hombro_3\n1,4750,38,90,hombro_3\n1,4772,37,90,hombro_3\n1,4792,37,90,hombro_3\n1,4813,36,90,hombro_3\n1,4833,35,90,hombro_3\n1,4855,35,90,hombro_3\n1,4875,34,90,hombro_3\n1,4896,33,90,hombro_3\n1,4916,33,90,hombro_3\n1,4938,32,90,hombro_3\n1,4958,32,90,hombro_3\n1,4979,32,90,hombro_3\n1,4999,31,90,hombro_3\n1,5021,31,90,hombro_3\n1,5041,30,90,hombro_3\n1,5062,30,90,hombro_3\n1,5083,30,90,hombro_3\n1,5104,30,90,hombro_3\n1,5124,30,90,hombro_3\n1,5144,30,90,hombro_3\n1,5166,30,90,hombro_3\n1,5186,30,90,hombro_3\n1,5207,30,90,pausa_3\n1,5227,30,90,pausa_3\n1,5248,30,90,pausa_3\n1,5268,30,90,pausa_3\n1,5289,30,90,pausa_3\n1,5310,30,90,pausa_3\n1,5331,30,90,pausa_3\n1,5351,30,90,pausa_3\n1,5372,30,90,pausa_3\n1,5392,30,90,pausa_3\n1,5413,30,90,pausa_3\n1,5433,30,90,pausa_3\n1,5454,30,90,pausa_3\n1,5474,30,90,pausa_3\n1,5496,30,90,pausa_3\n1,5507,30,90,pausa_3\n1,5508,30,90,codo_4\n1,5528,30,89,codo_4\n1,5549,30,89,codo_4\n1,5569,30,89,codo_4\n1,5591,30,89,codo_4\n1,5611,30,89,codo_4\n1,5632,30,88,codo_4\n1,5652,30,88,codo_4\n1,5674,30,87,codo_4\n1,5694,30,87,codo_4\n1,5715,30,86,codo_4\n1,5735,30,86,codo_4\n1,5757,30,85,codo_4\n1,5777,30,84,codo_4\n1,5798,30,84,codo_4\n1,5818,30,83,codo_4\n1,5840,30,82,codo_4\n1,5860,30,81,codo_4\n1,5881,30,80,codo_4\n1,5901,30,79,codo_4\n1,5923,30,78,codo_4\n1,5943,30,77,codo_4\n1,5964,30,76,codo_4\n1,5984,30,75,codo_4\n1,6006,30,74,codo_4\n1,6026,30,73,codo_4\n1,6047,30,72,codo_4\n1,6067,30,71,codo_4\n1,6089,30,69,codo_4\n1,6109,30,68,codo_4\n1,6130,30,67,codo_4\n1,6150,30,66,codo_4\n1,6172,30,65,codo_4\n1,6192,30,63,codo_4\n1,6213,30,62,codo_4\n1,6233,30,61,codo_4\n1,6254,30,60,codo_4\n1,6275,30,58,codo_4\n1,6295,30,57,codo_4\n1,6317,30,56,codo_4\n1,6337,30,55,codo_4\n1,6358,30,53,codo_4\n1,6378,30,52,codo_4\n1,6400,30,51,codo_4\n1,6420,30,50,codo_4\n1,6441,30,49,codo_4\n1,6461,30,48,codo_4\n1,6483,30,46,codo_4\n1,6503,30,45,codo_4\n1,6524,30,44,codo_4\n1,6544,30,43,codo_4\n1,6566,30,42,codo_4\n1,6586,30,41,codo_4\n1,6607,30,40,codo_4\n1,6627,30,39,codo_4\n1,6649,30,38,codo_4\n1,6669,30,37,codo_4\n1,6690,30,36,codo_4\n1,6710,30,36,codo_4\n1,6732,30,35,codo_4\n1,6752,30,34,codo_4\n1,6773,30,33,codo_4\n1,6793,30,33,codo_4\n1,6815,30,32,codo_4\n1,6835,30,32,codo_4\n1,6856,30,31,codo_4\n1,6876,30,31,codo_4\n1,6898,30,30,codo_4\n1,6918,30,30,codo_4\n1,6939,30,30,codo_4\n1,6959,30,30,codo_4\n1,6981,30,30,codo_4\n1,7001,30,30,codo_4\n1,7021,30,30,codo_4\n1,7042,30,30,pausa_4\n1,7062,30,30,pausa_4\n1,7084,30,30,pausa_4\n1,7104,30,30,pausa_4\n1,7125,30,30,pausa_4\n1,7145,30,30,pausa_4\n1,7166,30,30,pausa_4\n1,7186,30,30,pausa_4\n1,7207,30,30,pausa_4\n1,7227,30,30,pausa_4\n1,7248,30,30,pausa_4\n1,7268,30,30,pausa_4\n1,7290,30,30,pausa_4\n1,7310,30,30,pausa_4\n1,7331,30,30,pausa_4\n1,7342,30,30,pausa_4\n1,7343,30,30,fin\n'
PROCEDENCIA = {'origen': 'Ejecucion del nuevo sketch de hombro y codo en Wokwi; no es una captura del proyecto de base giratoria.', 'proyecto_usado_como_editor': 'https://wokwi.com/projects/477503605988146177', 'fecha_captura_utc': '2026-10-25T22:29:01.186786+00:00', 'metodo': 'Sesion temporal de Chrome: reemplazo del sketch y del diagrama, compilacion, simulacion y pulsacion del boton. No se guardaron cambios en el proyecto publicado.', 'filas': 363, 'significado': 'Consignas de posicion enviadas a Servo.write(); no mediciones fisicas.', 'sha256_sketch_ino': '03060102dea76a65f9a4ee096a7f6e04998bdc61b9e9eb69b1b6d221cfe146ea', 'sha256_diagram_json': '65fc96de095735235967c9f42c578bc67156f7cd330c1091c8299b14f3cf593d'}

In [ ]:
from IPython.display import HTML, display
if MODO == "captura_incluida":
    datos = cargar_csv(CSV_INCLUIDO, ciclo=CICLO, origen="Captura Wokwi · hombro y codo")
elif MODO == "subir_csv":
    from google.colab import files
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Subir un solo archivo CSV o TXT.")
    texto = next(iter(subidos.values())).decode("utf-8-sig")
    datos = cargar_csv(texto, ciclo=CICLO)
elif MODO == "pegar_csv":
    datos = cargar_csv(CSV_PEGADO, ciclo=CICLO)
elif MODO == "demostracion":
    datos = datos_demostracion()
else:
    raise ValueError("MODO no reconocido.")
resultado = verificar_independencia(datos)
print(json.dumps(resultado, ensure_ascii=False, indent=2))
if not resultado["configuraciones_en_orden"] or not resultado["regresa_a_reposo"]:
    print("ATENCIÓN: registro incompleto; se reproducen los datos disponibles.")

## 4. Ver el brazo

La animación indica qué articulación se mueve. Las curvas muestran que una consigna cambia mientras la otra permanece fija. La muñeca es el punto verde en el extremo del antebrazo.

El ángulo de cada servo se mantiene hasta la siguiente muestra. La reproducción no interpola un movimiento nuevo sobre el CSV y conserva las pausas registradas.

In [ ]:
fig, anim = crear_animacion(datos, fps=FPS)
guardar_html(anim, "brazo_interactivo.html", fps=FPS)
display(HTML(Path("brazo_interactivo.html").read_text(encoding="utf-8")))
plt.close(fig)

## 5. Exportar

El HTML contiene todos los cuadros y funciona sin internet. Para exportar también GIF o MP4, activar las opciones correspondientes. MP4 necesita FFmpeg disponible; HTML y GIF no lo requieren.

In [ ]:
EXPORTAR_GIF = False
EXPORTAR_MP4 = False
DESCARGAR_ARCHIVOS = False
archivos = ["brazo_interactivo.html"]
if EXPORTAR_GIF:
    anim.save("brazo.gif", writer=PillowWriter(fps=FPS), dpi=85)
    archivos.append("brazo.gif")
if EXPORTAR_MP4:
    import shutil
    if not shutil.which("ffmpeg"):
        raise RuntimeError("FFmpeg no está disponible: usar HTML o GIF.")
    anim.save("brazo.mp4", writer=FFMpegWriter(fps=FPS, bitrate=1800), dpi=100)
    archivos.append("brazo.mp4")
print("Archivos listos:", ", ".join(archivos))
if DESCARGAR_ARCHIVOS:
    from google.colab import files
    for nombre in archivos:
        files.download(nombre)

## 6. Código y conexionado para Wokwi

El hombro está conectado a D9; el codo, a D10; y el pulsador, a D2 y GND, con `INPUT_PULLUP`. Al arrancar se conecta y posiciona primero el hombro, se espera 1,5 s y luego se conecta y posiciona el codo. Tras otra espera de 1,5 s aparece `# LISTO` en el monitor serie.

Cada desplazamiento del ciclo se realiza llamando a una función que solo escribe en el servo recibido. El otro mantiene su consigna. Presionar el pulsador ejecuta el ciclo; mantenerlo presionado no lo repite.

Para otra captura: cargar los archivos del ZIP en un proyecto Arduino UNO de Wokwi, iniciar, esperar a `# LISTO`, presionar el botón y copiar el monitor serie hasta `# FIN_CICLO`, incluido el encabezado CSV. Luego elegir `subir_csv` o `pegar_csv` en la sección 3.

In [ ]:
# @title Preparar el ZIP del nuevo proyecto de Wokwi
SKETCH_ARDUINO = b'#include <Servo.h>\n\n// Grupo 3: Mateo Flores y Federico Soler.\n// Brazo planar: hombro + humero + codo + antebrazo. Una articulacion por vez.\nServo servoHombro;\nServo servoCodo;\nconst int PIN_BOTON = 2;\nconst unsigned long T = 1500;\nconst unsigned long PASO_MS = 20;\nconst int PUNTOS[5][2] = {{30, 30}, {75, 30}, {75, 90}, {30, 90}, {30, 30}};\nint anguloHombro = 30;\nint anguloCodo = 30;\nunsigned int ciclo = 0;\nunsigned long inicioCiclo;\n\nvoid registrar(const char *fase) {\n  // Son consignas de posicion, no mediciones de sensores.\n  Serial.print(ciclo);\n  Serial.print(\',\');\n  Serial.print(millis() - inicioCiclo);\n  Serial.print(\',\');\n  Serial.print(anguloHombro);\n  Serial.print(\',\');\n  Serial.print(anguloCodo);\n  Serial.print(\',\');\n  Serial.println(fase);\n}\n\nvoid setup() {\n  pinMode(PIN_BOTON, INPUT_PULLUP);\n  // Inicializacion escalonada: el codo sigue desconectado mientras se ubica\n  // el hombro. No se ordenan dos desplazamientos iniciales a la vez.\n  servoHombro.attach(9);\n  servoHombro.write(anguloHombro);\n  delay(T);\n  servoCodo.attach(10);\n  servoCodo.write(anguloCodo);\n  delay(T);\n  Serial.begin(115200);\n  Serial.println("ciclo,tiempo_ms,hombro_deg,codo_deg,fase");\n  Serial.println("# LISTO: presionar el pulsador");\n}\n\nvoid moverUnaArticulacion(Servo &motor, int &anguloActual, int destino,\n                          const char *fase) {\n  // Esta funcion solo escribe en el servo recibido. La otra consigna se mantiene.\n  const int origen = anguloActual;\n  unsigned long inicio = millis();\n  float s = 0;\n  while (s < 1.0) {\n    s = (millis() - inicio) / (float)T;\n    if (s > 1.0) s = 1.0;\n    float k = 3*s*s - 2*s*s*s;\n    anguloActual = origen + (destino - origen) * k;\n    motor.write(anguloActual);\n    registrar(fase);\n    delay(PASO_MS);\n  }\n}\n\nvoid pausaConRegistro(const char *fase) {\n  const unsigned long duracion = 300;\n  unsigned long inicio = millis();\n  while (millis() - inicio < duracion) {\n    registrar(fase);\n    unsigned long transcurrido = millis() - inicio;\n    if (transcurrido < duracion) {\n      unsigned long restante = duracion - transcurrido;\n      delay(restante < PASO_MS ? restante : PASO_MS);\n    }\n  }\n  registrar(fase);\n}\n\nvoid loop() {\n  if (digitalRead(PIN_BOTON) == LOW) {\n    ciclo++;\n    inicioCiclo = millis();\n    registrar("inicio");\n    moverUnaArticulacion(servoHombro, anguloHombro, PUNTOS[1][0], "hombro_1");\n    pausaConRegistro("pausa_1");\n    moverUnaArticulacion(servoCodo, anguloCodo, PUNTOS[2][1], "codo_2");\n    pausaConRegistro("pausa_2");\n    moverUnaArticulacion(servoHombro, anguloHombro, PUNTOS[3][0], "hombro_3");\n    pausaConRegistro("pausa_3");\n    moverUnaArticulacion(servoCodo, anguloCodo, PUNTOS[4][1], "codo_4");\n    pausaConRegistro("pausa_4");\n    registrar("fin");\n    Serial.println("# FIN_CICLO");\n    while (digitalRead(PIN_BOTON) == LOW) delay(10);\n  }\n}\n'
DIAGRAMA_WOKWI = b'{\r\n  "version": 1,\r\n  "author": "Mateo Flores, Federico Soler",\r\n  "editor": "wokwi",\r\n  "parts": [\r\n    {\r\n      "type": "wokwi-arduino-uno",\r\n      "id": "uno",\r\n      "top": 0,\r\n      "left": 0,\r\n      "attrs": {}\r\n    },\r\n    {\r\n      "type": "wokwi-servo",\r\n      "id": "servoHombro",\r\n      "top": -150,\r\n      "left": 300,\r\n      "attrs": {}\r\n    },\r\n    {\r\n      "type": "wokwi-servo",\r\n      "id": "servoCodo",\r\n      "top": -40,\r\n      "left": 300,\r\n      "attrs": {}\r\n    },\r\n    {\r\n      "type": "wokwi-pushbutton",\r\n      "id": "btn",\r\n      "top": 227,\r\n      "left": 297.6,\r\n      "attrs": {\r\n        "color": "green"\r\n      }\r\n    }\r\n  ],\r\n  "connections": [\r\n    [\r\n      "servoHombro:PWM",\r\n      "uno:9",\r\n      "orange",\r\n      []\r\n    ],\r\n    [\r\n      "servoHombro:V+",\r\n      "uno:5V",\r\n      "red",\r\n      []\r\n    ],\r\n    [\r\n      "servoHombro:GND",\r\n      "uno:GND.1",\r\n      "black",\r\n      []\r\n    ],\r\n    [\r\n      "servoCodo:PWM",\r\n      "uno:10",\r\n      "orange",\r\n      []\r\n    ],\r\n    [\r\n      "servoCodo:V+",\r\n      "uno:5V",\r\n      "red",\r\n      []\r\n    ],\r\n    [\r\n      "servoCodo:GND",\r\n      "uno:GND.1",\r\n      "black",\r\n      []\r\n    ],\r\n    [\r\n      "btn:1.l",\r\n      "uno:2",\r\n      "green",\r\n      []\r\n    ],\r\n    [\r\n      "btn:2.l",\r\n      "uno:GND.2",\r\n      "black",\r\n      []\r\n    ]\r\n  ],\r\n  "dependencies": {}\r\n}\r\n'
DESCARGAR_PROYECTO = False
import zipfile
with zipfile.ZipFile("Wokwi_Hombro_Codo_Independiente.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("sketch.ino", SKETCH_ARDUINO)
    z.writestr("diagram.json", DIAGRAMA_WOKWI)
    z.writestr("libraries.txt", "Servo\n")
    z.writestr("captura_wokwi.csv", CSV_INCLUIDO)
    z.writestr("procedencia.json", json.dumps(PROCEDENCIA, ensure_ascii=False, indent=2))
print("Creado: Wokwi_Hombro_Codo_Independiente.zip")
if DESCARGAR_PROYECTO:
    from google.colab import files
    files.download("Wokwi_Hombro_Codo_Independiente.zip")

## Alcance y referencias

La independencia comprobada se refiere a cambios en las **consignas articulares**. El registro no contiene mediciones de posición ni una verificación dinámica de un prototipo físico. La flexión fija del codo no impide que su posición en el espacio cambie al mover el hombro.

- Universidad de Palermo. *Trayectorias* e *Implementación de las trayectorias robóticas*, apuntes del módulo 6.
- [Monitor serie de Wokwi](https://docs.wokwi.com/guides/serial-monitor).
- [Servo de Wokwi](https://docs.wokwi.com/parts/wokwi-servo).
- [Animaciones de Matplotlib](https://matplotlib.org/stable/api/animation_api.html).